# Анализ датасета "Stroke Prediction"

## Общая информация

| Параметр | Значение |
|----------|----------|
| Источник | [Kaggle (fedesoriano, Stroke Prediction Dataset)](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset) |
| Исходный размер | 5110 строк, 12 колонок |
| Целевая переменная | `stroke` (0 - нет инсульта, 1 - инсульт) |
| Тип задачи | Бинарная классификация (принудительно указана, `FORCE_TASK_TYPE = 'classification'`) |
| Пропуски | 201 строка (3.9%), только в колонке `bmi` |
| Дисбаланс классов | 95.7% / 4.3% (соотношение 22:1) |

### Выполненные преобразования

| Действие | Результат |
|----------|-----------|
| Удаление колонки `id` | Убрана бесполезная информация |
| Приведение названий колонок к нижнему регистру | `Residence_type` → `residence_type` |
| Приведение значений категорий к нижнему регистру | `'Male'` → `'male'`, `'Yes'` → `'yes'` |
| Замена пробелов в категориях на подчёркивания | `'formerly smoked'` → `'formerly_smoked'` |

### Оптимизация типов данных

| Исходный тип | Новый тип | Колонки | Причина |
|--------------|-----------|---------|---------|
| `object` | `category` | gender, ever_married, work_type, residence_type, smoking_status | Экономия памяти, ускорение |
| `int64` | `Int64` | hypertension, heart_disease | Поддержка nullable integer |

### Итог после очистки

| Параметр | Значение |
|----------|----------|
| Размер после очистки | 4853 строки, 11 колонок |
| Пропуски | 0 |
| Числовые признаки | age, hypertension, heart_disease, avg_glucose_level, bmi |
| Категориальные признаки | gender, ever_married, work_type, residence_type, smoking_status |

---

## Пропуски

При анализе данных были обнаружены **пропуски** в колонке `bmi`: 201 строка, что составляет 3.9% от общего объёма данных. Пустые строки (`''` вместо NaN) и подозрительные значения (-1, -999, 999999) отсутствуют.

**Решение:** Доля пропусков <5%. Потеря 3.9% данных незначительна. Допустимо **удалить** строки с пропусками в bmi. Простое решение, не требует выбора стратегии заполнения.

**В шаблоне:** настройки в блоке 1.2 - `DROP_ROWS_WITH_NULLS = True`, `COLUMNS_TO_CHECK_NULLS = ['bmi']`

---


## Значение `'Unknown'` в smoking_status

При анализе категориальных признаков было обнаружено, что в колонке `smoking_status` присутствует значение `'Unknown'`.

**Важно:** это не пропуск и не скрытая пустота.  
Колонка не содержит `NaN`, пустых строк или маркеров пустоты. `'Unknown'` это полноценная категория: опрашиваемый не указал данные, а не пропустил их.

**Решение:** на этапе предобработки это значение **не удалялось и не заменялось**.  Выбор стратегии обработки `'Unknown'` отложен до этапа EDA.

---

## Подозрительные значения в возрасте

**Минимальное значение age 0.08:** 43 строки с возрастом < 1 года (0.08-0.88 лет). Во всех случаях stroke = 0, hypertension = 0, heart_disease = 0.

**Проверка:** Это реальные данные или ошибка данных? Можно ли удалить эту часть данных?

**Фактчекинг:** Младенческий или перинатальный инсульт бывает, но достаточно редко. Перинатальный инсульт возможен как раз до 28 дня жизни младенца (age 0.08). Часто не имеет типичных симптомов. Данные реальны, но в выборке нет ни одного stroke = 1. Эти данные не помогут научить модель прогнозировать инсульт.

**Решение:** удалить строки с возрастом < 1 года. Верхнюю границу age оставляем нетронутой (риск инсульта с возрастом увеличивается - это важно для предсказания). Удаление 0.8% данных не повлияет на модель. Допустимо.

**В шаблоне:** `NUMERIC_RANGES = {'age': (1, 82)}`

---

## Проверка значений в BMI

**bmi max = 97.6:** Максимальное значение 97.6. Реальное ожирение или ошибка ввода?

**Проверка:** bmi BMI 97.6 > 60: 13 строк. Во всех случаях stroke = 0. Максимальное значение bmi: age 17, гипертония 1, инсульт 0. При таких bmi дополнительные проблемы со здоровьем должны быть ярко выражены и преобладать. В 13 строках явной тенденции к дополнительным заболеваниям нет. Это могло бы создать ложное впечатление, что BMI > 60 не связан с дополнительными рисками, что противоречит реальности.

**Решение:** Удалить строки с BMI > 60. Это похоже на ошибки в данных. Удаление 0.25% строк некритично. Значения BMI до 60 оставлены.

**В шаблоне:** `NUMERIC_RANGES = {'bmi': (10, 60)}`

---

## Проверка значений колонки gender

В колонке gender обнаружено значение `Other`.

**Вопрос:** С медицинской точки зрения биологических полов два. Допустимо ли включение категории `Other` (скорее, социального-медицинского компонента) в медицинское прогнозирование? Есть ли исследования о распространенности инсульта у других гендеров.

**Проверка:** Категория `Other` представлена одной строкой. Это упрощает решение. Удалить, т.к. модель не сможет выучить связь по одному примеру. 1 строка это 0.02% от всех данных. Если бы выборка была значительной, то ее следовало бы оставить, т.к. есть исследования подтверждающие повышенный риск сердечно-сосудистых заболеваний у трансгендеров. В этом случае это была бы полезная информация.

**Решение:** Удалить строку с gender = 'Other'. Удаление одной строки (0.02%) некритично.

**В шаблоне:** `DROP_RARE_CATEGORIES = True`, `COLUMNS_TO_CHECK_RARE = ['gender']`, `RARE_CATEGORY_THRESHOLD = 0.01`

---

## Выбросы в глюкозе

IQR метод показывает 11.9% выбросов в avg_glucose_level. Верхняя граница по IQR - 165 мг/дл.

**Проверка:** распределение по процентилям: 50% (медиана) - 91.7 (норма), 90%  187.9, 95% - 215.4, 99% - 240.6. Медицинская норма глюкозы натощак  до 100 мг/дл, диабет - от 126. Верхние 10% выборки (глюкоза > 187) - это диабет. Удаление этих значений ухудшит предсказание модели.

**Решение:** Оставить значения выше 165 мг/дл. Высокая глюкоза - реальный фактор риска инсульта. Удаление "выбросов" приведёт к потере группы диабетиков.


---

## Выбросы в бинарных признаках

IQR метод показывает выбросы в hypertension (9.5%) и heart_disease (5.0%).

**Решение:** IQR-метод предполагает нормальное распределение и неприменим к бинарным признакам. Значения 1, которые встречаются реже 0, не являются выбросами - это просто редкий класс признака. Данные оставлены без изменений.

---

## Дисбаланс классов

**Дисбаланс классов** Распределение целевой переменной stroke: 95.7% - 0, 4.3% - 1. Соотношение 22:1. Модель может игнорировать редкий класс, если не принять меры.

**Решение:** При обучении использовать `class_weight='balanced'` или SMOTE. Без коррекции дисбаланса модель не выучит признаки класса 1.

---

## Корреляции и мультиколлинеарность

### Корреляции

Сильных корреляций (>0.7) **нет**. Максимальная корреляция: age - bmi (0.32).

### VIF

**VIF всех числовых признаков < 2**

| Признак | VIF |
|---------|-----|
| age | 1.27 |
| bmi | 1.14 |
| hypertension | 1.12 |
| avg_glucose_level | 1.10 |
| heart_disease | 1.09 |

**Решение:** Использовать все 5 числовых признаков. Мультиколлинеарность отсутствует, дублирования информации нет.

---

## Константные колонки

Константных колонок (только одно значение) **не обнаружено**.

Колонка heart_disease имеет низкую вариативность (95% значений - 0), но оставлена как значимый медицинский признак.

---

## Категориальные признаки и рекомендации по кодированию

**Факт:**

| Колонка | Уникальных значений | Рекомендация |
|---------|---------------------|--------------|
| gender | 2 | LabelEncoder |
| ever_married | 2 | LabelEncoder |
| residence_type | 2 | LabelEncoder |
| work_type | 5 | OneHotEncoder (drop='first') |
| smoking_status | 4 | OneHotEncoder (drop='first') |

**Решение:** Бинарные признаки кодировать LabelEncoder, многоклассовые - OneHotEncoder с drop='first'.

---

## Предварительный выбор моделей

Анализ данных позволяет предположить, какие модели будут эффективны:

| Модель | Требования | Эффективность |
|--------|------------|-------------------------|
| Логистическая регрессия | Масштабирование, кодирование | Средняя |
| RandomForest | Кодирование (OrdinalEncoder) | Высокая (устойчив к выбросам) |
| XGBoost | Кодирование, настройка гиперпараметров | Потенциально высокая |

**Стартовая модель:** RandomForest с `class_weight='balanced'` (учитывает дисбаланс, устойчива к выбросам).

---

## Итоговые файлы

Все файлы сохраняются в Colab в папке `/content/{PROJECT_NAME}/`

| Категория | Файлы |
|-----------|-------|
| Исходные данные | `00_raw.csv` |
| Контрольные точки | `checkpoints/01_structure.csv`, `checkpoints/02_cleaned.csv` |
| Выборки | `splits/X_train.csv`, `splits/X_val.csv`, `splits/X_test.csv`, `splits/y_train.csv`, `splits/y_val.csv`, `splits/y_test.csv` |
| Метаданные | `metadata.json` |
| Отчёты | `reports/data_report.txt`, `reports/analysis_report.txt` |

---

## Выводы

1. Данные очищены от пропусков и явных аномалий. Всего удалено 256 строк (5.01%).
2. Выбросы в глюкозе не удалялись: диабетики, ключевая группа риска.
3. Выбросы в бинарных признаках: неиформативно, игнорируются.
4. Обнаружен сильный дисбаланс классов (22:1): требует коррекции при обучении.
5. Мультиколлинеарность отсутствует (VIF < 2), все признаки можно использовать.
6. Категориальные признаки требуют кодирования (LabelEncoder для бинарных, OneHotEncoder для многоклассовых).

---

## Следующие шаги

1. Кодирование категориальных признаков (согласно рекомендациям)
2. Масштабирование числовых признаков (для линейных моделей)
3. Обучение моделей с `class_weight='balanced'`
4. Оценка качества по метрикам ROC-AUC и F1-score